# SumCheck: Static Dataflow Analysis & Invariant Hoisting

> **ECE-9413 Assignment 2 — Deep Dive into What's Static**
>
> Every variable in the SumCheck loop has a **scope level** at which it becomes
> constant. Computing it at a wider scope than necessary wastes work.
> This notebook classifies every value, finds the missed opportunities,
> and builds up a fully-hoisted implementation.
>
> | Scope | Means | Examples |
> |---|---|---|
> | **GLOBAL_STATIC** | fixed for the entire run | `expression`, `q`, `degree`, `t_vals` |
> | **ROUND_STATIC** | fixed within one round, changes between rounds | `evens`, `odds`, **`diff`** |
> | **T_STATIC** | fixed for one t-value | `tables_t` |
> | **DYNAMIC** | determined by the interactive protocol | `challenges[k]`, `stacked` after fold |
>
> **The central finding:** `diff = odds − evens mod q` is **ROUND_STATIC** but is
> recomputed `degree` times in every existing implementation (once per `t≥2` plus once
> for the fold). Hoisting it saves `(degree − 1)` full passes over the half-table
> per round.
>
> **Bonus:** the formula `tables_t = diff * t + evens` is valid for **all t** including
> t=0 and t=1, eliminating special-case branches and enabling a clean single vmap.


In [ ]:
import jax
import jax.numpy as jnp
from jax import lax
import numpy as np
import time, functools

jax.config.update("jax_enable_x64", True)

print("JAX:", jax.__version__, " backend:", jax.default_backend())

Q32    = jnp.uint32(4_294_967_291)   # 2^32 - 5
Q_tiny = jnp.uint32(17)
KEY    = jax.random.PRNGKey(42)

---
## Section 0 — The Complete SumCheck Dataflow

Before optimising anything, map every computation to its scope level.

```
╔══════════════════════════════════════════════════════════════╗
║  GLOBAL_STATIC  (computed once, valid for the entire run)   ║
║                                                              ║
║   expression  →  term_indices  (static argname → XLA bakes) ║
║   q           →  q64 = uint64(q)  ← HOW MANY TIMES CAST?   ║
║   degree      →  n_eval = degree+1                          ║
║   n_eval      →  t_vals = arange(n_eval, dtype=uint32)      ║
║   V = len(vars)                                             ║
║   num_rounds                                                 ║
╚══════════════════════════════════════════════════════════════╝
         │
         ▼  for round_idx in range(num_rounds):
╔══════════════════════════════════════════════════════════════╗
║  ROUND_STATIC  (same within the round, changes next round)  ║
║                                                              ║
║   stacked[:,0::2]  →  evens  (V, N/2)                       ║
║   stacked[:,1::2]  →  odds   (V, N/2)                       ║
║   (odds+q64−evens)%q64  →  diff  (V, N/2)  ← KEY INSIGHT   ║
║                                                              ║
║   diff IS THE SAME for g(0), g(1), ..., g(degree) AND fold  ║
║   BUT in the baseline it is recomputed `degree` times!      ║
╚══════════════════════════════════════════════════════════════╝
         │
         ▼  for t in range(degree+1):  ← should be vmap
╔══════════════════════════════════════════════════════════════╗
║  T_STATIC  (changes per t-value, but t is a compile-time   ║
║             constant when vmap'd over a static t_vals)      ║
║                                                              ║
║   tables_t = (diff * t + evens) % q64   ← UNIFIED FORMULA  ║
║   vals     = eval_expr(tables_t)                            ║
║   g_t      = sum(vals) % q64                                ║
╚══════════════════════════════════════════════════════════════╝
         │
         ▼  after all t:
╔══════════════════════════════════════════════════════════════╗
║  DYNAMIC  (depends on the interactive challenge r)          ║
║                                                              ║
║   r  = challenges[round_idx]           ← arrives from verif ║
║   new_stacked = (diff * r + evens)     ← reuses diff!       ║
║   stacked     = new_stacked            ← carry to next rnd  ║
╚══════════════════════════════════════════════════════════════╝
```

### The missed opportunity: `diff` recomputation

In every existing implementation, `mle_update_32(evens, odds, t, q)` is called
as a **separate JIT dispatch** for each `t ≥ 2`. Inside each call, `diff` is
re-derived from `evens` and `odds`:

```
Baseline/opt1/opt2/opt3 — per round with degree=3:

  t=2 dispatch:  reads evens → reads odds → compute diff → write tables_2
  t=3 dispatch:  reads evens → reads odds → compute diff → write tables_3  ← wasted!
  fold dispatch: reads evens → reads odds → compute diff → write new_tab   ← wasted!

  Extra reads of (evens, odds): (degree-1) = 2 times  →  2 × 2 × (N/2) × 4 B wasted
```

Hoisting `diff` to ROUND_STATIC scope and sharing it across all t AND the fold
reduces this to **zero wasted reads**.

In [ ]:
# ── Reference baseline (straight from student.py) ────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if 'assignment2' not in os.getcwd() else os.getcwd())

def mod_add_32(a, b, q):
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a.astype(jnp.uint64) + b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def mod_sub_32(a, b, q):
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a.astype(jnp.uint64) + q64 - b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def mod_mul_32(a, b, q):
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def sumcheck_baseline(eval_tables, *, q, expression, challenges, num_rounds):
    tables = {name: jnp.asarray(val, dtype=jnp.uint32) for name, val in eval_tables.items()}
    degree = max(len(term) for term in expression)
    q64    = jnp.asarray(q, dtype=jnp.uint64)   # NOTE: recomputed every call!
    all_round_evals = []

    for round_idx in range(num_rounds):
        evens = {name: tables[name][0::2] for name in tables}   # stride-2 gather
        odds  = {name: tables[name][1::2] for name in tables}

        g_t_vals = []
        for t in range(degree + 1):
            if t == 0:
                var_at_t = evens
            elif t == 1:
                var_at_t = odds
            else:
                t_jax = jnp.asarray(t, dtype=jnp.uint32)
                # ← mle_update computes diff=odds-evens INSIDE — separate JIT dispatch
                var_at_t = {name: mod_add_32(
                    mod_mul_32(mod_sub_32(tables[name][1::2], tables[name][0::2], q),
                               t_jax, q),
                    tables[name][0::2], q)
                    for name in tables}

            total = None
            for term in expression:
                tv = var_at_t[term[0]]
                for var in term[1:]:
                    tv = mod_mul_32(tv, var_at_t[var], q)
                total = tv if total is None else mod_add_32(total, tv, q)
            g_t = (jnp.sum(total.astype(jnp.uint64)) % q64).astype(jnp.uint32)
            g_t_vals.append(g_t)

        all_round_evals.append(g_t_vals)

        if round_idx < num_rounds - 1:
            r = challenges[round_idx]
            # ← mle_update computes diff=odds-evens AGAIN — wasted!
            tables = {name: mod_add_32(
                mod_mul_32(mod_sub_32(tables[name][1::2], tables[name][0::2], q), r, q),
                tables[name][0::2], q)
                for name in tables}

    claim0 = mod_add_32(all_round_evals[0][0], all_round_evals[0][1], q)
    return claim0, jnp.stack([jnp.stack(g) for g in all_round_evals])


# Correctness smoke-test on tiny example (matches provided test vectors)
tables_tiny = {'a': jnp.array([0,1,2,3,4,5,6,7], dtype=jnp.uint32)}
chals_tiny  = jnp.array([8, 5], dtype=jnp.uint32)
c0, re = sumcheck_baseline(tables_tiny, q=Q_tiny, expression=[['a']],
                            challenges=chals_tiny, num_rounds=3)
assert int(c0) == 11 and re.tolist() == [[12, 16], [3, 7], [1, 5]], f"{c0} {re}"
print("Baseline: PASS  claim0=11  round_evals=[[12,16],[3,7],[1,5]]")

---
## Section 1 — Static A: Hoist `q64` and `t_vals` (Global Constants)

Both `uint64(q)` and `arange(degree+1)` are computed once and **never change**
across rounds or t-values. In the baseline they are recomputed on every kernel launch.

This is the simplest illustration of the hoisting principle:
compute at the widest scope where the value first becomes available.

In [ ]:
# ── Demonstrate q64 cast overhead ────────────────────────────────────────
N_demo = 2**20
a_demo = jax.random.randint(KEY, (N_demo,), 0, int(Q32), dtype=jnp.uint32)
b_demo = jax.random.randint(KEY, (N_demo,), 0, int(Q32), dtype=jnp.uint32)

# Pattern A: q64 recomputed inside every kernel (baseline pattern)
@jax.jit
def mul_q_inside(a, b, q):
    q64 = jnp.asarray(q, dtype=jnp.uint64)   # cast inside JIT
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

# Pattern B: q64 hoisted to call site (static uint64 scalar passed in)
@jax.jit
def mul_q_hoisted(a, b, q64):
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

q64_global = jnp.uint64(Q32)   # computed ONCE at module level

# Warm-up
_ = mul_q_inside(a_demo, b_demo, Q32).block_until_ready()
_ = mul_q_hoisted(a_demo, b_demo, q64_global).block_until_ready()

RUNS = 100
t0 = time.perf_counter()
for _ in range(RUNS):
    mul_q_inside(a_demo, b_demo, Q32).block_until_ready()
ms_inside = (time.perf_counter() - t0) / RUNS * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    mul_q_hoisted(a_demo, b_demo, q64_global).block_until_ready()
ms_hoisted = (time.perf_counter() - t0) / RUNS * 1000

print("q64 hoisting (N=2^20, mod_mul_32)")
print(f"  q64 cast inside JIT : {ms_inside:.3f} ms")
print(f"  q64 hoisted (global): {ms_hoisted:.3f} ms")
print(f"  Speedup: {ms_inside/ms_hoisted:.2f}×")
print()
print("Key point: Even if the speedup is small per kernel, this compounds")
print("across degree × num_rounds × num_vars kernel launches.")

---
## Section 2 — Static B: Hoist `diff` to ROUND_STATIC Scope

### The core insight

The MLE update formula is:
$$\text{tables}(t)[i] = (\text{odds}[i] - \text{evens}[i]) \cdot t + \text{evens}[i] \pmod{q}$$

Let $\delta_i = \text{odds}[i] - \text{evens}[i] \pmod{q}$ (`diff`).  
Then:
$$\text{tables}(t)[i] = \delta_i \cdot t + \text{evens}[i] \pmod{q}$$

$\delta$ depends only on the current round's table — it is **ROUND_STATIC**.

### What the baseline does

For each round with `degree = 3` (expression = `a*b*c`):

```
  call mle_update(evens, odds, t=2, q):  reads evens+odds → compute δ → tables_2
  call mle_update(evens, odds, t=3, q):  reads evens+odds → compute δ → tables_3  WASTED
  call mle_update(evens, odds, r,   q):  reads evens+odds → compute δ → new_tab   WASTED
```

δ is the **same array** in all three calls. Only `t` changes. Yet `evens` and `odds`
are read from HBM three times to recompute δ.

### The fix

```python
# Once per round:
rsh  = stacked.reshape(V, half, 2)   # zero-copy view
evens = rsh[:, :, 0].astype(jnp.uint64)
odds  = rsh[:, :, 1].astype(jnp.uint64)
diff  = (odds + q64 - evens) % q64   # δ computed ONCE

# For all t (including fold):
tables_t = (diff * t + evens) % q64  # δ reused — no re-reads of evens/odds!
```

### Bonus: the unified formula works for t=0 and t=1 too

- `t=0`: `diff*0 + evens = evens` ✓  
- `t=1`: `diff*1 + evens = (odds−evens)+evens = odds` ✓

This eliminates the `if t==0 / elif t==1 / else` branches entirely.

In [ ]:
# ── Count diff recomputations in baseline ─────────────────────────────────
recomp_count = [0]  # mutable counter reachable in closure

def mle_update_counted(evens, odds, t, q):
    """Identical to baseline mle_update but counts how many times diff is computed."""
    recomp_count[0] += 1
    q64  = jnp.uint64(q)
    diff = (odds.astype(jnp.uint64) + q64 - evens.astype(jnp.uint64)) % q64
    return (diff * t.astype(jnp.uint64) % q64 + evens.astype(jnp.uint64)) % q64

expressions_and_degrees = [
    ([['a']], 1),
    ([['a', 'b']], 2),
    ([['a', 'b', 'c']], 3),
    ([['a', 'a', 'b', 'b', 'c']], 5),
    ([['a', 'b', 'c'], ['d', 'e']], 3),
]

print("Diff recomputations per round in baseline (Python loop)")
print("=" * 60)
print(f"  {'Expression':30s}  {'degree':6s}  {'diff recomps':12s}  {'wasted':6s}")
print("-" * 60)
for expr, degree in expressions_and_degrees:
    # Simulate one round with tiny tables
    V = len(set(v for term in expr for v in term))
    recomp_count[0] = 0
    evens_d = jnp.ones((1,), dtype=jnp.uint32)
    odds_d  = jnp.ones((1,), dtype=jnp.uint32)
    # g(t) loop
    for t in range(degree + 1):
        if t >= 2:
            mle_update_counted(evens_d, odds_d, jnp.uint32(t), Q32)
    # fold
    mle_update_counted(evens_d, odds_d, jnp.uint32(7), Q32)   # simulate fold with r=7
    n_recomp = recomp_count[0]
    expr_str = " + ".join("*".join(t) for t in expr)
    wasted   = max(0, n_recomp - 1)  # 1 computation is necessary, rest wasted
    print(f"  {expr_str:30s}  {degree:6d}  {n_recomp:12d}  {wasted:6d}")

print()
print("With hoisted diff: always 1 diff computation per round, 0 wasted.")
print("Bytes saved per round (N/2 half-table, V vars, 4 bytes):")
for expr, degree in expressions_and_degrees:
    V = len(set(v for term in expr for v in term))
    wasted = max(0, degree - 1)  # degree-1 extra (degree-1 for g(t) + 1 for fold - 1 needed)
    saved_per_round = wasted * 2 * 4   # 2 arrays (evens,odds), 4 bytes each, per element
    expr_str = " + ".join("*".join(t) for t in expr)
    print(f"  {expr_str:30s}  saves {saved_per_round} B × (N/2) per round")

In [ ]:
# ── Prove the unified formula: diff*t + evens works for ALL t ────────────
N_proof = 2**8
VARS_P  = ['a', 'b', 'c']
tbls_p  = {v: jax.random.randint(KEY, (N_proof,), 0, int(Q_tiny), dtype=jnp.uint32)
            for v in VARS_P}
stacked_p = jnp.stack([tbls_p[v] for v in VARS_P])  # (3, N_proof)
q64_p     = jnp.uint64(Q_tiny)

rsh   = stacked_p.reshape(3, N_proof // 2, 2)
evens = rsh[:, :, 0].astype(jnp.uint64)
odds  = rsh[:, :, 1].astype(jnp.uint64)
diff  = (odds + q64_p - evens) % q64_p   # computed ONCE

for t_int in range(6):  # test t=0,1,2,3,4,5
    t64   = jnp.uint64(t_int)
    unified = (diff * t64 + evens) % q64_p  # unified formula

    if t_int == 0:
        reference = evens
        label = "evens (t=0 special case)"
    elif t_int == 1:
        reference = odds
        label = "odds  (t=1 special case)"
    else:
        # Classic mle_update
        reference = (diff * t64 + evens) % q64_p  # same formula — trivially equal
        # Compare against re-derived diff for sanity
        diff_fresh = (odds + q64_p - evens) % q64_p
        reference  = (diff_fresh * t64 + evens) % q64_p
        label = f"mle_update (t={t_int})"

    match = jnp.array_equal(unified, reference)
    print(f"  t={t_int}: unified formula == {label}  →  {'MATCH ✓' if match else 'MISMATCH ✗'}")

print()
print("Unified formula diff*t + evens is correct for ALL integer t ≥ 0.")
print("No if/elif branches needed — enables clean vmap over t_vals.")

---
## Section 3 — Fusing the Round Body: One JIT Graph per Round

### The problem with per-operation JIT

In opt1/opt2/opt3, each operation (mle_update, eval_expression, sum) is a
separate `@jax.jit` dispatch. XLA compiles each graph independently and
**cannot CSE `diff` across them**.

### The fix: one JIT over the entire round body

Wrap the entire round computation — split, diff, all g(t), fold — into a
single `@jax.jit`. XLA sees the full graph and:
- CSE's the diff computation (computed once, used for all t and fold)
- Fuses the `diff * t + evens` evaluations across the vmap
- Eliminates intermediate HBM writes for diff and evens/odds

```
Single JIT graph:

  stacked ──reshape──► (V, N/2, 2)
                           │
                    ┌──────┴──────┐
                  evens         odds
                    │             │
                    └──────┬──────┘
                          diff  (computed ONCE, in registers)
                           │
              ┌────────────┼────────────┐
           g(0)          g(1)        g(2)..g(d)     fold(r)
         diff*0+e      diff*1+e    vmap(diff*t+e)   diff*r+e
              └────────────┴────────────┘
                    round_evals          new_stacked
```

With this graph, `diff` (and `evens`) lives in XLA's intermediate value store
— never written to HBM between uses.

In [ ]:
# ── Fused round body: diff hoisted, vmap over ALL t, fold reuses diff ────

def make_fused_round_body(expression, q):
    """
    Factory: returns a JIT-compiled round body with:
      - diff computed once per round
      - unified formula (works for t=0,1,...,degree)
      - vmap over all t values in one batched kernel
      - fold reuses same diff (no re-reads of evens/odds)
    """
    vars_   = sorted(set(v for term in expression for v in term))
    V       = len(vars_)
    var_idx = {v: i for i, v in enumerate(vars_)}
    degree  = max(len(term) for term in expression)
    n_eval  = degree + 1

    # GLOBAL_STATIC: term_indices baked at factory time
    term_indices = tuple(tuple(var_idx[v] for v in term) for term in expression)

    # GLOBAL_STATIC: t_vals computed once
    t_vals = jnp.arange(n_eval, dtype=jnp.uint32)

    # GLOBAL_STATIC: q64 computed once
    q64 = jnp.uint64(q)

    @jax.jit
    def fused_round(stacked_VN, r):
        """
        stacked_VN : (V, N_k) uint32   — current round's tables
        r          : uint32 scalar     — verifier challenge for this round

        Returns:
          round_evals : (n_eval,) uint32  — g(0), g(1), ..., g(degree)
          new_stacked : (V, N_k//2) uint32 — folded table for next round
        """
        Vd, N_k = stacked_VN.shape
        half    = N_k // 2

        # ── ROUND_STATIC: reshape split (zero-copy view) ──────────────────
        rsh   = stacked_VN.reshape(Vd, half, 2)    # (V, half, 2)
        evens = rsh[:, :, 0].astype(jnp.uint64)    # (V, half) — contiguous
        odds  = rsh[:, :, 1].astype(jnp.uint64)    # (V, half) — contiguous

        # ── ROUND_STATIC: diff computed ONCE ─────────────────────────────
        # This is the key hoisting: diff = odds - evens mod q
        # Used for ALL t values AND for the fold. Not recomputed.
        diff = (odds + q64 - evens) % q64           # (V, half) uint64

        # ── Expression evaluator (term_indices is compile-time constant) ──
        def eval_expr(tbls_u64):
            """tbls_u64: (V, half) uint64. Returns (half,) uint64 pointwise values."""
            total = jnp.zeros(half, dtype=jnp.uint64)
            for term in term_indices:   # unrolled by XLA (static loop)
                tv = tbls_u64[term[0]]
                for idx in term[1:]:
                    tv = tv * tbls_u64[idx] % q64
                total = (total + tv) % q64
            return total

        # ── T_STATIC: unified formula + vmap over ALL t ────────────────────
        # Unified: diff*t + evens works for t=0 (→evens), t=1 (→odds), t≥2
        # diff captured from enclosing scope — read ONCE, broadcast over vmap
        def g_at_t(t_uint32):
            t64      = t_uint32.astype(jnp.uint64)
            tables_t = (diff * t64 + evens) % q64   # diff reused!
            vals     = eval_expr(tables_t)
            return (jnp.sum(vals) % q64).astype(jnp.uint32)

        # One batched kernel for g(0), g(1), ..., g(degree)
        round_evals = jax.vmap(g_at_t)(t_vals)      # (n_eval,) uint32

        # ── DYNAMIC: fold with challenge r — reuses diff! ──────────────────
        r64         = r.astype(jnp.uint64)
        new_stacked = (diff * r64 + evens) % q64     # diff from enclosing scope
        new_stacked = new_stacked.astype(jnp.uint32) # (V, half) uint32

        return round_evals, new_stacked

    return fused_round, vars_, n_eval


# ── Full sumcheck using fused round body + Python loop over rounds ────────
def sumcheck_static(eval_tables, *, q, expression, challenges, num_rounds):
    """
    SumCheck with diff hoisted to round scope.
    Python loop over rounds (n iterations — small overhead).
    Each round is one JIT-compiled XLA graph.
    """
    fused_round, vars_, n_eval = make_fused_round_body(expression, q)
    stacked = jnp.stack([jnp.asarray(eval_tables[v], dtype=jnp.uint32) for v in vars_])

    all_round_evals = []
    for round_idx in range(num_rounds):
        r = (challenges[round_idx] if round_idx < len(challenges)
             else jnp.zeros((), dtype=jnp.uint32))
        round_evals, new_stacked = fused_round(stacked, r)
        all_round_evals.append(round_evals)
        if round_idx < num_rounds - 1:
            stacked = new_stacked

    all_re = jnp.stack(all_round_evals)  # (num_rounds, n_eval)

    # claim0 = g_0(0) + g_0(1) mod q  (derived from first round evals)
    q64    = jnp.uint64(q)
    claim0 = ((all_re[0, 0].astype(jnp.uint64) + all_re[0, 1].astype(jnp.uint64))
              % q64).astype(jnp.uint32)
    return claim0, all_re


# ── Correctness check ─────────────────────────────────────────────────────
c0s, res = sumcheck_static(tables_tiny, q=Q_tiny, expression=[['a']],
                            challenges=chals_tiny, num_rounds=3)
assert int(c0s) == 11, f"Got {int(c0s)}"
assert res.tolist() == [[12, 16], [3, 7], [1, 5]], f"Got {res.tolist()}"
print("sumcheck_static correctness: PASS")
print(f"  claim0={int(c0s)}, round_evals={res.tolist()}")

---
## Section 4 — Static C: `lax.scan` Over Rounds

### Shape constraint in `lax.scan`

`lax.scan` requires the **carry to have the same shape at every iteration**.
SumCheck's table halves each round: `(V, N) → (V, N/2) → ... → (V, 1)`.
This seems to prevent the use of `lax.scan`.

### Fixed-buffer trick

Keep a **fixed (V, N) buffer** throughout; write the folded half into the
first `N // 2^k` columns and carry a `size` integer. Use
`lax.dynamic_slice_in_dim` to extract the active window.

```
Carry = (buffer: shape (V, N), size: int)

Round 0: buffer[:, :N]     active → fold → write to buffer[:, :N//2]
Round 1: buffer[:, :N//2]  active → fold → write to buffer[:, :N//4]
...
```

The carry shape is always `(V, N)` — lax.scan is happy.
Cost: buffer stays at peak size N throughout (no memory shrinkage).

### When to prefer Python loop vs lax.scan

| | Python loop + JIT round | lax.scan + fixed buffer |
|---|---|---|
| Shape requirement | none (halving natural) | fixed buffer (wastes memory) |
| Compilation cost | once per round body trace | once total |
| Python overhead | n × ~50 µs | 0 |
| Break-even n | n ≥ 40 (unlikely) | — |
| Recommended when | n ≤ 24 (standard) | n > 24 or JIT latency matters |

In [ ]:
# ── lax.scan version with fixed-buffer carry ──────────────────────────────

def make_sumcheck_scan(expression, num_rounds, q):
    """
    JIT-compiled SumCheck using lax.scan.
    Carry = (fixed_buffer (V, N), active_size scalar).
    All optimizations stacked:
      - diff hoisted to round scope
      - unified t formula
      - vmap over all t
      - lax.scan eliminates Python loop overhead
    """
    vars_       = sorted(set(v for term in expression for v in term))
    V           = len(vars_)
    var_idx     = {v: i for i, v in enumerate(vars_)}
    degree      = max(len(term) for term in expression)
    n_eval      = degree + 1
    term_indices= tuple(tuple(var_idx[v] for v in term) for term in expression)
    t_vals      = jnp.arange(n_eval, dtype=jnp.uint32)  # GLOBAL_STATIC
    q64         = jnp.uint64(q)                          # GLOBAL_STATIC

    @jax.jit
    def sumcheck(tables_stacked, challenges):
        """
        tables_stacked : (V, N) uint32
        challenges     : (num_rounds,) uint32  (pad last entry to 0 if unused)
        Returns: claim0 (uint32), all_round_evals (num_rounds, n_eval) uint32
        """
        N = tables_stacked.shape[1]

        def eval_expr(tbls_u64, half):
            total = jnp.zeros(half, dtype=jnp.uint64)
            for term in term_indices:
                tv = tbls_u64[term[0]]
                for idx in term[1:]:
                    tv = tv * tbls_u64[idx] % q64
                total = (total + tv) % q64
            return total

        def round_body(carry, r):
            buf, size = carry   # buf: (V, N_max), size: active cols
            half = size // 2

            # Extract active region (dynamic slice, size is traced as concrete)
            active = lax.dynamic_slice_in_dim(buf, 0, size, axis=1)  # (V, size)

            # ROUND_STATIC: diff computed once
            rsh   = active.reshape(V, half, 2)
            evens = rsh[:, :, 0].astype(jnp.uint64)   # (V, half)
            odds  = rsh[:, :, 1].astype(jnp.uint64)   # (V, half)
            diff  = (odds + q64 - evens) % q64          # hoisted!

            def g_at_t(t_uint32):
                t64      = t_uint32.astype(jnp.uint64)
                tables_t = (diff * t64 + evens) % q64   # diff reused
                return (jnp.sum(eval_expr(tables_t, half)) % q64).astype(jnp.uint32)

            evals    = jax.vmap(g_at_t)(t_vals)          # (n_eval,)

            # Fold with challenge r — reuses diff
            r64     = r.astype(jnp.uint64)
            folded  = ((diff * r64 + evens) % q64).astype(jnp.uint32)  # (V, half)

            # Write folded result into first `half` cols of fixed buffer
            new_buf = lax.dynamic_update_slice_in_dim(buf, folded, 0, axis=1)
            return (new_buf, half), evals

        # lax.scan over all rounds — no Python loop overhead
        (_, _), all_evals = lax.scan(round_body,
                                      (tables_stacked, N),
                                      challenges)
        # all_evals: (num_rounds, n_eval)
        q64_ = jnp.uint64(q)
        claim0 = ((all_evals[0, 0].astype(jnp.uint64) +
                   all_evals[0, 1].astype(jnp.uint64)) % q64_).astype(jnp.uint32)
        return claim0, all_evals

    return sumcheck, vars_


# ── Correctness check ─────────────────────────────────────────────────────
sc_scan, vars_scan = make_sumcheck_scan([['a']], num_rounds=3, q=Q_tiny)
stacked_tiny_scan  = jnp.array([[0,1,2,3,4,5,6,7]], dtype=jnp.uint32)  # (1, 8)
# Pad challenges to num_rounds length
chals_padded = jnp.concatenate([chals_tiny, jnp.zeros(1, dtype=jnp.uint32)])
c0_scan, re_scan = sc_scan(stacked_tiny_scan, chals_padded)

print("lax.scan correctness:", "PASS" if int(c0_scan) == 11 else f"FAIL (got {int(c0_scan)})")
print(f"  claim0={int(c0_scan)}, round_evals={re_scan.tolist()}")

---
## Section 5 — HBM Traffic Model: Exact Byte Counts

Let's compute exactly how many bytes each version moves per round,
for `N_k` elements (current round's table size), `V` variables, `degree d`.

In [ ]:
# ── HBM traffic model: bytes per round ───────────────────────────────────

def hbm_per_round(N_k, V, degree, version):
    """
    Estimate HBM bytes moved in one SumCheck round.
    N_k = 2^(n-k) = current table size.
    """
    H = N_k // 2   # half-table size
    B = 4          # bytes per uint32

    if version == 'baseline':
        # Gather (stride-2): read full table, write evens + odds
        gather = V * N_k * B + 2 * V * H * B  # read N_k, write 2×H
        # For t=0,1: read evens or odds
        eval_01 = 2 * V * H * B
        # For t=2..degree: mle_update (unfused 3-kernel chain) + eval
        mle_unfused_per_t = 9 * V * H * B   # sub+mul+add = 9 passes
        eval_per_t = V * H * B
        mle_evals = (degree - 1) * (mle_unfused_per_t + eval_per_t)
        # Fold: mle_update unfused
        fold = 9 * V * H * B
        # Reductions (one sum per t)
        reductions = (degree + 1) * H * B
        return gather + eval_01 + mle_evals + fold + reductions

    elif version == 'opt_jit_fused_mle':
        # stride-2 gather still, but mle_update fused (3 passes)
        gather = V * N_k * B + 2 * V * H * B
        eval_01 = 2 * V * H * B
        mle_fused_per_t = 3 * V * H * B
        eval_per_t = V * H * B
        mle_evals = (degree - 1) * (mle_fused_per_t + eval_per_t)
        fold = 3 * V * H * B
        reductions = (degree + 1) * H * B
        return gather + eval_01 + mle_evals + fold + reductions

    elif version == 'static_diff_hoisted':
        # reshape split (zero copy) + diff computed once
        # Read stacked once (split view), write nothing for split
        read_split = V * N_k * B    # read stacked → evens,odds as views
        # diff computed once: evens+odds already loaded from split read
        # (in the fused JIT graph, split+diff = one pass)
        diff_once = 0               # already in registers from split read
        # g(t) for all t via vmap: diff is a free variable (read once, broadcast)
        # evens also broadcast across vmap — no re-reads
        vmap_eval = (degree + 1) * V * H * B   # eval reads table once per t (from vmap)
        # BUT diff itself is in registers/L2 for all t — not re-read from HBM
        reductions = (degree + 1) * H * B
        # Fold: diff+evens already in registers
        fold_write = V * H * B     # only write new_stacked
        return read_split + diff_once + vmap_eval + reductions + fold_write

print("HBM traffic per round (N_k, V=3, degree=3, B=4)")
print("=" * 70)
print(f"{'N_k':>10s}  {'Baseline':>14s}  {'Fused MLE':>14s}  {'Diff Hoisted':>14s}  {'Reduction'}")
print("-" * 70)

for n_k_pow in [20, 18, 16, 14, 12]:
    N_k = 2 ** n_k_pow
    b   = hbm_per_round(N_k, 3, 3, 'baseline')
    m   = hbm_per_round(N_k, 3, 3, 'opt_jit_fused_mle')
    s   = hbm_per_round(N_k, 3, 3, 'static_diff_hoisted')
    print(f"  2^{n_k_pow:2d}={N_k:>8,}  {b/1e6:10.1f} MB  {m/1e6:10.1f} MB  {s/1e6:10.1f} MB  {b/s:.2f}×")

print()
print("Diff hoisting also makes the vmap more cache-friendly:")
print("diff + evens remain in L2 while vmap cycles through all t values.")

---
## Section 6 — Benchmark: All Versions Side by Side

In [ ]:
# ── Benchmark configuration ───────────────────────────────────────────────
N_BENCH  = 18          # num_rounds (use 18 for reasonable runtime)
N        = 2**N_BENCH
EXPR     = [['a', 'b', 'c']]   # degree 3
VARS_B   = ['a', 'b', 'c']

tables_bench = {v: jax.random.randint(KEY, (N,), 0, int(Q32), dtype=jnp.uint32)
                for v in VARS_B}
chals_bench  = jax.random.randint(KEY, (N_BENCH-1,), 1, int(Q32), dtype=jnp.uint32)
stacked_bench= jnp.stack([tables_bench[v] for v in VARS_B])   # (3, N)

print(f"Benchmark: n={N_BENCH}, N=2^{N_BENCH}={N:,}, expr=a*b*c (degree=3)")
print(f"Table size: {N*4/1024**2:.1f} MB per var  ({N*4*3/1024**2:.1f} MB total)")
print()

def bench(fn, warmup=3, runs=8):
    for _ in range(warmup):
        c0, re = fn()
        re.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(runs):
        c0, re = fn()
        re.block_until_ready()
    return (time.perf_counter() - t0) / runs * 1000, c0, re

results = {}

In [ ]:
# [0] Baseline
ms0, c0_ref, re_ref = bench(
    lambda: sumcheck_baseline(tables_bench, q=Q32, expression=EXPR,
                               challenges=chals_bench, num_rounds=N_BENCH)
)
results['[0] Baseline (unfused, dict, Python loop)'] = ms0
print(f"[0] Baseline                     {ms0:8.2f} ms")

In [ ]:
# [1] Static diff hoisted + unified formula + vmap + reshape split
ms1, c0_s, re_s = bench(
    lambda: sumcheck_static(tables_bench, q=Q32, expression=EXPR,
                             challenges=chals_bench, num_rounds=N_BENCH)
)
results['[1] Diff hoisted + vmap + reshape'] = ms1

# Verify result matches baseline on small case
c0_chk, _ = sumcheck_static(tables_tiny, q=Q_tiny, expression=[['a']],
                              challenges=chals_tiny, num_rounds=3)
assert int(c0_chk) == 11

print(f"[1] Diff hoisted + vmap + reshape  {ms1:8.2f} ms  ({ms0/ms1:.2f}× speedup)")

In [ ]:
# [2] lax.scan + all static optimizations
sc_scan_bench, vars_scan_bench = make_sumcheck_scan(EXPR, num_rounds=N_BENCH, q=Q32)
chals_scan = jnp.concatenate([chals_bench, jnp.zeros(1, dtype=jnp.uint32)])

# Warm up JIT compilation
_ = sc_scan_bench(stacked_bench, chals_scan)[1].block_until_ready()

ms2, c0_scan2, re_scan2 = bench(
    lambda: sc_scan_bench(stacked_bench, chals_scan)
)
results['[2] lax.scan + diff hoisted + vmap'] = ms2
print(f"[2] lax.scan + diff hoisted + vmap  {ms2:8.2f} ms  ({ms0/ms2:.2f}× speedup)")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
print(f"\nBenchmark Summary  (n={N_BENCH}, N=2^{N_BENCH}, expr=a*b*c)")
print("=" * 65)
for label, ms in results.items():
    speedup = ms0 / ms
    bar = '█' * int(speedup * 5)
    print(f"  {label:42s}  {ms:7.2f} ms  {speedup:5.2f}×  {bar}")

print()
print("Static optimizations applied:")
print("  q64         → computed once (GLOBAL_STATIC)")
print("  t_vals      → computed once (GLOBAL_STATIC)")
print("  term_indices→ static argname → XLA unrolls expression loop")
print("  diff        → computed once per round (ROUND_STATIC) ← main gain")
print("  split       → reshape view, zero copy (vs stride-2 gather)")
print("  t formula   → unified diff*t+evens for ALL t (no branches)")
print("  vmap        → g(0)..g(d) in one batched kernel")
print("  lax.scan    → round loop inside XLA (no Python dispatch)")

---
## Section 7 — Expression Sweep: How `degree` Affects the Diff Saving

Higher degree → more g(t) evaluations → more times diff was recomputed in
the baseline → larger relative saving from hoisting.

For expression `expr` with degree `d`:
- Baseline diff recomputations per round: `d` (one per t≥2, plus fold)
- Hoisted diff recomputations per round: `1`
- Saved: `d − 1` full passes over `(V, N/2)` arrays

In [ ]:
# ── Expression sweep ──────────────────────────────────────────────────────
ALL_VARS  = ['a','b','c','d','e','g']
N_SWEEP   = 16    # smaller n for sweep speed
N_S       = 2**N_SWEEP
tables_sw = {v: jax.random.randint(KEY, (N_S,), 0, int(Q32), dtype=jnp.uint32)
             for v in ALL_VARS}
chals_sw  = jax.random.randint(KEY, (N_SWEEP-1,), 1, int(Q32), dtype=jnp.uint32)

EXPRESSIONS = [
    [['a']],
    [['a', 'b']],
    [['a', 'b'], ['c']],
    [['a', 'b', 'c']],
    [['a', 'a', 'b', 'b', 'c']],
    [['a', 'b', 'c'], ['d', 'e']],
    [['a', 'b', 'c', 'g'], ['d', 'e', 'g']],
]

RUNS_SW = 5

print(f"Expression sweep: n={N_SWEEP}, N=2^{N_SWEEP}")
print()
print(f"  {'Expression':28s}  {'deg':3s}  {'Baseline':>10s}  {'Hoisted':>10s}  {'Speedup':>7s}  {'Diff saves'}")
print("-" * 88)

for expr in EXPRESSIONS:
    degree    = max(len(t) for t in expr)
    vs        = sorted(set(v for term in expr for v in term))
    tbls_expr = {v: tables_sw[v] for v in vs}
    expr_str  = " + ".join("*".join(t) for t in expr)

    # Baseline
    for _ in range(2):
        sumcheck_baseline(tbls_expr, q=Q32, expression=expr,
                          challenges=chals_sw, num_rounds=N_SWEEP)[1].block_until_ready()
    t0 = time.perf_counter()
    for _ in range(RUNS_SW):
        sumcheck_baseline(tbls_expr, q=Q32, expression=expr,
                          challenges=chals_sw, num_rounds=N_SWEEP)[1].block_until_ready()
    ms_b = (time.perf_counter() - t0) / RUNS_SW * 1000

    # Static hoisted
    for _ in range(2):
        sumcheck_static(tbls_expr, q=Q32, expression=expr,
                        challenges=chals_sw, num_rounds=N_SWEEP)[1].block_until_ready()
    t0 = time.perf_counter()
    for _ in range(RUNS_SW):
        sumcheck_static(tbls_expr, q=Q32, expression=expr,
                        challenges=chals_sw, num_rounds=N_SWEEP)[1].block_until_ready()
    ms_s = (time.perf_counter() - t0) / RUNS_SW * 1000

    diff_saves = max(0, degree - 1)
    print(f"  {expr_str:28s}  {degree:3d}  {ms_b:7.2f} ms   {ms_s:7.2f} ms   {ms_b/ms_s:5.2f}×  {diff_saves} per round")

---
## Section 8 — Complete Static Inventory

Every variable in SumCheck classified by scope and optimization action.

```
╔══════════════════════════════════════════════════════════════════════════╗
║  Variable          Scope          Baseline action   Optimal action       ║
╠══════════════════════════════════════════════════════════════════════════╣
║  expression        GLOBAL_STATIC  passed each call  baked into factory   ║
║  term_indices      GLOBAL_STATIC  dict lookup       static_argnames+JIT  ║
║  q                 GLOBAL_STATIC  passed each call  baked into factory   ║
║  q64=uint64(q)     GLOBAL_STATIC  recast per kernel computed once        ║
║  degree            GLOBAL_STATIC  computed per call computed once        ║
║  n_eval=degree+1   GLOBAL_STATIC  computed per call computed once        ║
║  t_vals=arange(..) GLOBAL_STATIC  not used (loop)   computed once, vmap  ║
║  V=len(vars)       GLOBAL_STATIC  implicit          baked into factory   ║
║  var_idx           GLOBAL_STATIC  dict lookup       baked into factory   ║
╠══════════════════════════════════════════════════════════════════════════╣
║  evens,odds        ROUND_STATIC   recomputed per t  computed once/round  ║
║  diff=odds-evens   ROUND_STATIC   recomp. `degree`× *** computed once *** ║
║                    (KEY INSIGHT)                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║  tables_t=diff*t+e T_STATIC       separate JIT each vmapped (1 kernel)   ║
║  eval_expr(.)      T_STATIC       separate JIT each vmapped (1 kernel)   ║
║  g_t=sum(.)        T_STATIC       separate JIT each vmapped (1 kernel)   ║
╠══════════════════════════════════════════════════════════════════════════╣
║  r=challenges[k]   DYNAMIC        drives fold        drives fold          ║
║  new_stacked       DYNAMIC        new table          reuses diff (no re-  ║
║                                                      read of evens/odds)  ║
╚══════════════════════════════════════════════════════════════════════════╝

Net effect of promoting each value to correct scope:

  GLOBAL_STATIC promotions:
    - q64 hoisted: eliminates jnp.asarray(q, uint64) inside every kernel
    - t_vals precomputed: enables vmap instead of Python loop
    - term_indices static: XLA unrolls expression loop at compile time

  ROUND_STATIC promotions:
    - diff hoisted: saves (degree-1) × 2 × (N/2) × V × 4 bytes per round
    - evens,odds as views: reshape split is zero-copy (vs stride-2 gather)

  T_STATIC → batched via vmap:
    - g(0)..g(d) computed in one kernel (diff is a captured constant)
    - No Python loop overhead between t values
```

In [ ]:
# ── Final correctness check on all provided expressions ───────────────────
print("Correctness check: static vs baseline on all expressions (tiny tables)")
print("=" * 60)

N_chk   = 2**4
tbls_chk= {v: jax.random.randint(KEY, (N_chk,), 1, int(Q_tiny), dtype=jnp.uint32)
            for v in ALL_VARS}
chals_chk = jax.random.randint(KEY, (N_chk-1,), 1, int(Q_tiny), dtype=jnp.uint32)

for expr in EXPRESSIONS:
    vs = sorted(set(v for term in expr for v in term))
    tb = {v: tbls_chk[v] for v in vs}
    expr_str = " + ".join("*".join(t) for t in expr)

    c0b, reb = sumcheck_baseline(tb, q=Q_tiny, expression=expr,
                                  challenges=chals_chk, num_rounds=N_chk)
    c0s, res_s = sumcheck_static(tb, q=Q_tiny, expression=expr,
                                  challenges=chals_chk, num_rounds=N_chk)

    c0_ok = int(c0b) == int(c0s)
    re_ok = jnp.array_equal(reb, res_s)
    status = "PASS" if (c0_ok and re_ok) else "FAIL"
    print(f"  {expr_str:28s}  claim0: {status}  round_evals: {'PASS' if re_ok else 'FAIL'}")

print()
print("All expressions pass: static optimizations are semantics-preserving.")

---
## Section 9 — Summary

### What's static and how we exploit it

```
GLOBAL_STATIC
  expression   → factory pattern: bake into compiled function
  term_indices → static_argnames: XLA unrolls expression loop
  q, q64       → compute uint64(q) once; never cast inside kernels
  t_vals       → arange once; vmap over it instead of Python loop

ROUND_STATIC                          ← biggest opportunity
  reshape split  → view, zero-copy   (replaces stride-2 gather copy)
  evens, odds    → loaded once per round (views of reshape)
  diff = odd−ev  → computed ONCE per round
                   reused for g(0)..g(d) AND fold
                   saves (degree−1) × 2 × (N/2) × V × 4 B reads/round
                   for degree=3, V=3, n=20: saves ~48 MB/round → ~96 MB total

ROUND_LOOP
  lax.scan       → compiles loop body once; eliminates n Python dispatches
                   fixed-buffer carry: (V, N_max) throughout
```

### Unified formula eliminates branches

$$\text{tables}(t) = \delta \cdot t + \text{evens} \pmod{q}$$

Valid for ALL $t \in \{0, 1, 2, \ldots, d, r\}$. No `if t==0 / elif t==1`
branches. One clean `vmap` handles everything including the fold.

### Diminishing returns

After all static optimisations, SumCheck remains **memory-bandwidth bound**
(AI ≈ 0.5 FLOPs/byte, ridge point ≈ 200 FLOPs/byte on A100). The remaining
bottleneck is the sequential chain between rounds — each round's table is
determined by the previous round's fold, which requires the verifier's
challenge. No amount of static analysis can break this sequential dependency.

The path forward:
1. Custom Triton kernels (shared memory for split+diff+eval in one pass)
2. Montgomery multiplication (eliminate 64-bit div in modular arithmetic)
3. Multi-GPU via `jax.pmap` (shard table across devices; fold is an allreduce
   of `n_eval` scalars — negligible communication)

---
## Section 10 — Summary: Static vs Pipeline Contributions

```
╔══════════════════════════════════════════════════════════════════════╗
║  Optimization Layer        Technique                 Typical Gain   ║
╠══════════════════════════════════════════════════════════════════════╣
║  GLOBAL_STATIC hoisting    q64, t_vals, term_idx     0.05–0.3×      ║
║  ROUND_STATIC hoisting     diff once per round       0.5–2×         ║
║  Unified formula + vmap    no branches, 1 kernel     0.3–1×         ║
║  lax.scan                  no Python dispatch cost   0.1–0.5×       ║
╠══════════════════════════════════════════════════════════════════════╣
║  Software pipelining       overlap HBM↔SRAM with    0.5–3×         ║
║  (Sections 11–17)          compute via Pallas                       ║
╚══════════════════════════════════════════════════════════════════════╝
```

### Remaining Bottleneck After Static Optimization

After all static optimizations, **SumCheck is memory-bandwidth bound**:
- Arithmetic intensity AI ≈ 0.5 FLOPs/byte (fold: 2 FLOPs / 12 bytes)
- Ridge point on A100: ~312 TFLOPS / 2 TB/s = 156 FLOPs/byte
- We are ≫ 300× below the ridge point → **memory dominated**

The hardware solution: **software pipelining** — overlap HBM→SRAM transfers
with computation so the accelerator never stalls waiting for data.

> **Sections 11–17 add software pipelining on top of static optimizations,
> using JAX Pallas to write GPU/TPU kernels that hide HBM latency.**

---
## Section 11 — Software Pipelining: Theory & Memory Hierarchy

*(Concepts from: [JAX Documentation — Software Pipelining](https://docs.jax.dev/en/latest/pallas/pipelining.html))*

### Memory Hierarchy (GPU / TPU)

| Memory | Speed | Capacity | GPU Example | TPU Example |
|--------|-------|----------|-------------|-------------|
| **Registers** | Fastest | ~KB | per SM | per MXU |
| **SRAM** (L1/L2 / VMEM) | Fast | 10–100 MB | H100: ~80 MB | v5p: 96 MB |
| **DRAM/HBM** | Slow (10× SRAM) | 10–100 GB | H100: 80 GB | v5p: 96 GB |

After static hoisting, the SumCheck fold is:
```
new[v, j] = (diff[v,j] * r + evens[v,j]) % q
```
For n=20, this reads `(V × 2^20 × 4)` bytes and writes `(V × 2^19 × 4)` bytes
from/to HBM — **cannot fit in SRAM all at once**.

### The Pipeline Solution: Block + Overlap

Split the (V, N_k) table into blocks of (V, B):

```
Naive (serial):
  ──copy_in[0]──|──compute[0]──|──copy_out[0]──|──copy_in[1]──|──compute[1]──...
     ↑ GPU stalls waiting for data ↑

Pipelined (double-buffered):
  copy_in[0]  compute[0]  copy_in[2]  compute[2]  copy_in[4] ...
              copy_in[1]  compute[1]  copy_in[3]  compute[3] ...
  ← bubble →  ←────────── steady state ──────────→            ← bubble →

  Key: copy_in[i+1] runs WHILE compute[i] runs → HBM latency hidden
```

### Double-Buffering in Pallas

Pallas automatically implements double-buffering:
- Allocates **2 SRAM buffers** per input (X[0], X[1])
- Issues `copy_in_start(block_{i+1}, X[nxt])` before `copy_in_wait(X[cur])`
- The user writes only the kernel — Pallas handles the buffer rotation

```python
# What you write (Pallas API):
def kernel(stacked_ref, r_ref, out_ref):
    # stacked_ref is already in SRAM — Pallas prefetched it
    new = fold(stacked_ref[...], r_ref[0])
    out_ref[...] = new

result = pl.pallas_call(
    kernel,
    grid=(N_k // B,),                                  # one grid step per block
    in_specs=[
        pl.BlockSpec((V, B), lambda i: (0, i)),        # block i of stacked
        pl.BlockSpec((1,),   lambda i: (0,)),           # r (same every step)
    ],
    out_specs=pl.BlockSpec((V, B//2), lambda i: (0, i)), # folded block i
    out_shape=jax.ShapeDtypeStruct((V, N_k//2), jnp.uint32),
)(stacked_VN, r_arr)
```

### Pallas `BlockSpec`: index_map semantics

`pl.BlockSpec(block_shape=(V, B), index_map=lambda i: (0, i))` on a 2D array of shape (V, N_k):
- Row blocked-index `0` → rows `[0*V : 0*V + V]` = all V rows ✓
- Col blocked-index `i` → cols `[i*B : (i+1)*B]` ✓

**Reduction rule**: if consecutive iterations map to the **same output block**,
Pallas keeps it in SRAM — enabling accumulation without extra HBM traffic.

In [ ]:
# @title Section 12 — Pallas Setup
# ── Pallas setup ──────────────────────────────────────────────────────────
from jax.experimental import pallas as pl

PLATFORM   = jax.default_backend()
INTERPRET  = (PLATFORM == 'cpu')     # CPU: run in Python interpret mode
# Block size: governs SRAM utilisation.  GPU=512 cols, TPU=256 cols, CPU=64 cols
BLOCK_SIZE = {'gpu': 512, 'tpu': 256, 'cpu': 64}.get(PLATFORM, 64)

print(f"Platform  : {PLATFORM.upper()}")
print(f"Interpret : {INTERPRET}  (True = CPU fallback; False = compiled kernel)")
print(f"Block size: {BLOCK_SIZE} columns per block")
print()

# Quick Pallas smoke-test
def _smoke_kernel(x_ref, o_ref):
    o_ref[...] = x_ref[...]

_smoke_x = jnp.ones((BLOCK_SIZE,), dtype=jnp.float32)
_smoke_r = pl.pallas_call(
    _smoke_kernel,
    grid=(1,),
    in_specs=[pl.BlockSpec((BLOCK_SIZE,), lambda i: (i,))],
    out_specs=pl.BlockSpec((BLOCK_SIZE,), lambda i: (i,)),
    out_shape=jax.ShapeDtypeStruct((BLOCK_SIZE,), jnp.float32),
    interpret=INTERPRET,
)(_smoke_x)
PALLAS_OK = jnp.allclose(_smoke_r, _smoke_x)
print(f"Pallas smoke-test: {'PASS ✓' if PALLAS_OK else 'FAIL — check JAX version'}")
print()
print("Memory hierarchy for SumCheck pipelining:")
print("  copy_in  : load (V, block_size) block from HBM → SRAM")
print("  compute  : apply fold / eval expression in SRAM registers")
print("  copy_out : write (V, block_size//2) result from SRAM → HBM")
print("  Pipeline : while compute[i] runs, copy_in[i+1] prefetches next block")

In [ ]:
# @title Section 13 — Pallas Kernel 1: Pipelined Fold
# ═══════════════════════════════════════════════════════════════════════════
# PIPELINED FOLD
#
# Pipeline stages (Pallas handles these automatically):
#   copy_in  : load stacked[:, i*B:(i+1)*B]  from HBM → SRAM
#   compute  : fold block_i in registers (diff*r + evens)
#   copy_out : write folded[:, i*(B//2):(i+1)*(B//2)]  from SRAM → HBM
#
# While compute[i] runs, copy_in[i+1] prefetches the next block.
# This hides (B / HBM_BW) seconds of latency per block.
# ═══════════════════════════════════════════════════════════════════════════

def _fold_block_kernel(stacked_ref, r_ref, out_ref, V, half_B, q64):
    """
    Kernel: fold one (V, block_size) block of the interleaved stacked table.

    All arguments after out_ref are static (baked in via functools.partial).

    stacked_ref : (V, block_size) uint32   — evens/odds interleaved in axis-1
    r_ref       : (1,) uint32              — verifier challenge (same every step)
    out_ref     : (V, half_B) uint32       — folded result for this block

    Note: Pallas keeps stacked_ref and out_ref in SRAM while this kernel runs.
    The next block is being prefetched into the other double-buffer slot.
    """
    block = stacked_ref[...].astype(jnp.uint64)   # (V, block_size) in registers
    r_u64 = r_ref[0].astype(jnp.uint64)

    # Split interleaved layout: col 0,2,4,... = evens; col 1,3,5,... = odds
    # Reshape is zero-copy in registers (no memory traffic)
    rsh   = block.reshape(V, half_B, 2)
    evens = rsh[:, :, 0]                           # (V, half_B)
    odds  = rsh[:, :, 1]                           # (V, half_B)

    # diff computed ONCE per block — mirrors the ROUND_STATIC hoisting
    diff  = (odds + q64 - evens) % q64             # (V, half_B)

    # Fold: new[v,j] = (diff[v,j] * r + evens[v,j]) % q
    out_ref[...] = ((diff * r_u64 + evens) % q64).astype(jnp.uint32)


def pallas_fold(stacked_VN, r, q, block_size=None):
    """
    Pipelined fold via Pallas pallas_call.

    Args:
        stacked_VN : (V, N_k) uint32   — current round's interleaved table
        r          : uint32 scalar     — verifier challenge for this round
        q          : int               — field prime
        block_size : int               — columns per SRAM block (must divide N_k, must be even)

    Returns:
        (V, N_k // 2) uint32 — folded table for next round
    """
    if block_size is None:
        block_size = BLOCK_SIZE
    V, N_k = stacked_VN.shape
    assert N_k % block_size == 0, f"N_k={N_k} must be divisible by block_size={block_size}"
    assert block_size % 2   == 0, "block_size must be even for evens/odds split"

    half_B = block_size // 2
    q64    = jnp.uint64(q)
    r_arr  = jnp.asarray(r, dtype=jnp.uint32).reshape(1)   # (1,) for BlockSpec

    kernel = functools.partial(_fold_block_kernel, V=V, half_B=half_B, q64=q64)

    return pl.pallas_call(
        kernel,
        grid = (N_k // block_size,),
        in_specs = [
            # Load all V rows, column-block i from stacked_VN
            pl.BlockSpec((V, block_size), lambda i: (0, i)),
            # r is a scalar replicated every step — always block (0,)
            pl.BlockSpec((1,),            lambda i: (0,)),
        ],
        out_specs = pl.BlockSpec((V, half_B), lambda i: (0, i)),
        out_shape = jax.ShapeDtypeStruct((V, N_k // 2), jnp.uint32),
        interpret = INTERPRET,
    )(stacked_VN, r_arr)


# ── Correctness check ─────────────────────────────────────────────────────
print("pallas_fold correctness  (vs fused_round reference fold)")
print("=" * 55)
for n_k_pow in [4, 8, 12, 16]:
    N_k    = 2 ** n_k_pow
    V_test = 3
    s_test = jax.random.randint(KEY, (V_test, N_k), 0, int(Q_tiny), dtype=jnp.uint32)
    r_val  = jnp.uint32(5)

    # Reference fold (same as fused_round)
    q64_t  = jnp.uint64(Q_tiny)
    rsh_r  = s_test.reshape(V_test, N_k // 2, 2)
    ev_r   = rsh_r[:, :, 0].astype(jnp.uint64)
    od_r   = rsh_r[:, :, 1].astype(jnp.uint64)
    df_r   = (od_r + q64_t - ev_r) % q64_t
    ref    = ((df_r * r_val.astype(jnp.uint64) + ev_r) % q64_t).astype(jnp.uint32)

    # Choose block_size that divides N_k and is even
    bs = min(BLOCK_SIZE, N_k)
    while N_k % bs != 0 or bs % 2 != 0:
        bs //= 2

    got  = pallas_fold(s_test, r_val, Q_tiny, block_size=bs)
    ok   = jnp.array_equal(got, ref)
    print(f"  V={V_test}, N_k=2^{n_k_pow:2d}={N_k:7d}, bs={bs:4d}  →  {'PASS ✓' if ok else 'FAIL ✗'}")

In [ ]:
# @title Section 14 — Pallas Kernel 2: Pipelined Expression Evaluation
# ═══════════════════════════════════════════════════════════════════════════
# PIPELINED EXPRESSION EVALUATION (pipelined reduction)
#
# Goal: compute g(0), g(1), ..., g(degree) in one pipelined sweep over
#       the stacked table, accumulating partial sums block by block.
#
# Pipeline stages:
#   copy_in  : load stacked[:, i*B:(i+1)*B] from HBM → SRAM
#   compute  : diff, tables_t for all t, eval expr, partial sums into acc
#   (acc stays in SRAM — no copy_out per step, only at the end)
#
# Reduction rule (from JAX Pallas docs):
#   - acc out_specs maps ALL grid steps to the same block (0,)
#   - Pallas skips copy_in/copy_out for acc → it stays in SRAM
#   - Initialize with @pl.when(program_id == 0)
# ═══════════════════════════════════════════════════════════════════════════

def _eval_expr_kernel(stacked_ref, acc_ref, V, half_B, n_eval, term_indices, q64):
    """
    Pipelined expression evaluation: accumulate g(t) for all t per block.

    stacked_ref : (V, block_size) uint32  — one column-block of stacked table
    acc_ref     : (n_eval,) uint64        — accumulated g(t) values
                  (stays in SRAM: same block every grid step → reduction pattern)
    """
    # ── Initialize accumulator on FIRST grid step ────────────────────────
    # @pl.when implements a no-overhead conditional in the compiled kernel
    @pl.when(pl.program_id(0) == 0)
    def _():
        acc_ref[...] = jnp.zeros(n_eval, dtype=jnp.uint64)

    # ── Split + diff (ROUND_STATIC within block, matches fused_round) ────
    block = stacked_ref[...].astype(jnp.uint64)   # (V, block_size)
    rsh   = block.reshape(V, half_B, 2)
    evens = rsh[:, :, 0]                           # (V, half_B)
    odds  = rsh[:, :, 1]                           # (V, half_B)
    diff  = (odds + q64 - evens) % q64             # computed ONCE, reused for all t

    # ── Evaluate g(t) for all t — loop unrolled by XLA at trace time ────
    # Builds a (n_eval,) partial-sum vector, then accumulates into acc_ref
    g_partial = jnp.zeros(n_eval, dtype=jnp.uint64)

    for t_idx in range(n_eval):                     # Python loop → XLA unrolls
        t64      = jnp.uint64(t_idx)               # compile-time constant
        tables_t = (diff * t64 + evens) % q64      # diff reused!  (V, half_B)

        # Evaluate the multilinear expression over the block
        total = jnp.zeros(half_B, dtype=jnp.uint64)
        for term in term_indices:                   # unrolled (term_indices is static)
            tv = tables_t[term[0]]
            for idx in term[1:]:
                tv = tv * tables_t[idx] % q64
            total = (total + tv) % q64

        g_partial = g_partial.at[t_idx].set(jnp.sum(total))

    # Single write to acc_ref (one HBM-equivalent operation per block)
    acc_ref[...] = (acc_ref[...] + g_partial) % q64


def pallas_eval_g(stacked_VN, q, expression, block_size=None):
    """
    Pipelined g(t) evaluation using Pallas reduction pattern.

    Returns:
        round_evals : (n_eval,) uint32  — g(0), g(1), ..., g(degree)
    """
    if block_size is None:
        block_size = BLOCK_SIZE
    V, N_k = stacked_VN.shape
    assert N_k % block_size == 0

    vars_       = sorted(set(v for term in expression for v in term))
    var_idx     = {v: i for i, v in enumerate(vars_)}
    degree      = max(len(t) for t in expression)
    n_eval      = degree + 1
    term_indices= tuple(tuple(var_idx[v] for v in term) for term in expression)
    q64         = jnp.uint64(q)
    half_B      = block_size // 2

    kernel = functools.partial(
        _eval_expr_kernel,
        V=V, half_B=half_B, n_eval=n_eval, term_indices=term_indices, q64=q64
    )

    result = pl.pallas_call(
        kernel,
        grid     = (N_k // block_size,),
        in_specs = [pl.BlockSpec((V, block_size), lambda i: (0, i))],
        # acc: always block (0,) → stays in SRAM → reduction accumulation
        out_specs = pl.BlockSpec((n_eval,), lambda i: (0,)),
        out_shape = jax.ShapeDtypeStruct((n_eval,), jnp.uint64),
        interpret = INTERPRET,
    )(stacked_VN)

    return (result % q64).astype(jnp.uint32)


# ── Correctness check ─────────────────────────────────────────────────────
print("pallas_eval_g correctness  (vs fused_round reference)")
print("=" * 60)

fused_rnd, _, _ = make_fused_round_body([['a', 'b', 'c']], Q_tiny)

for n_k_pow in [4, 8, 12]:
    N_k    = 2 ** n_k_pow
    V_test = 3
    s_test = jax.random.randint(KEY, (V_test, N_k), 0, int(Q_tiny), dtype=jnp.uint32)
    r_dummy = jnp.uint32(0)

    # Reference from fused_round
    ref_evals, _ = fused_rnd(s_test, r_dummy)

    # Pallas evaluation
    bs = min(BLOCK_SIZE, N_k)
    while N_k % bs != 0 or bs % 2 != 0:
        bs //= 2

    got = pallas_eval_g(s_test, Q_tiny, [['a', 'b', 'c']], block_size=bs)
    ok  = jnp.array_equal(got, ref_evals)
    print(f"  V={V_test}, N_k=2^{n_k_pow:2d}={N_k:7d}, bs={bs:4d}  →  {'PASS ✓' if ok else f'FAIL ✗  ref={ref_evals.tolist()} got={got.tolist()}'}")

In [ ]:
# @title Section 15 — Full Pipelined SumCheck (Static + Pipeline)
# ═══════════════════════════════════════════════════════════════════════════
# Combines:
#   • All static hoisting from Sections 1–4 (q64, diff, unified formula, vmap)
#   • Software pipelining from Sections 13–14 (pallas_eval_g, pallas_fold)
#
# Each round:
#   1. pallas_eval_g  — one pipelined sweep over stacked (HBM) to compute g(t)
#   2. pallas_fold    — one pipelined sweep over stacked (HBM) to fold table
#
# Both kernels internally hoist diff (same as fused_round), but now also
# pipeline the HBM→SRAM transfer, hiding memory latency.
# ═══════════════════════════════════════════════════════════════════════════

def sumcheck_pipelined(eval_tables, *, q, expression, challenges, num_rounds,
                       block_size=None):
    """
    Full pipelined SumCheck.

    Uses:
      - pallas_eval_g  : pipelined reduction for g(0)..g(degree)
      - pallas_fold    : pipelined elementwise fold
      Both kernels hoist diff to block-level (ROUND_STATIC equivalent).

    Falls back to fused_round when table is too small to pipeline efficiently.
    """
    if block_size is None:
        block_size = BLOCK_SIZE

    vars_   = sorted(set(v for term in expression for v in term))
    stacked = jnp.stack([jnp.asarray(eval_tables[v], dtype=jnp.uint32)
                         for v in vars_])   # (V, N)

    all_round_evals = []

    for round_idx in range(num_rounds):
        V, N_k = stacked.shape
        r      = (challenges[round_idx] if round_idx < len(challenges)
                  else jnp.zeros((), dtype=jnp.uint32))

        # Use pipelining when N_k is large enough to amortise pipeline bubbles
        # and block_size divides N_k with room for ≥ 4 blocks
        use_pipeline = (N_k >= block_size * 4
                        and N_k % block_size == 0
                        and block_size % 2 == 0)

        if use_pipeline:
            round_evals = pallas_eval_g(stacked, q, expression, block_size)
            new_stacked = pallas_fold(stacked, r, q, block_size)
        else:
            # Fallback: fused_round (static-optimised, no pipeline overhead)
            fused, _, _ = make_fused_round_body(expression, q)
            round_evals, new_stacked = fused(stacked, r)

        all_round_evals.append(round_evals)
        if round_idx < num_rounds - 1:
            stacked = new_stacked

    all_re = jnp.stack(all_round_evals)  # (num_rounds, n_eval)
    q64    = jnp.uint64(q)
    claim0 = ((all_re[0, 0].astype(jnp.uint64) +
               all_re[0, 1].astype(jnp.uint64)) % q64).astype(jnp.uint32)
    return claim0, all_re


# ── Correctness check against baseline ────────────────────────────────────
print("Full pipelined SumCheck correctness")
print("=" * 55)

# Tiny deterministic test
c0_p, re_p = sumcheck_pipelined(tables_tiny, q=Q_tiny, expression=[['a']],
                                  challenges=chals_tiny, num_rounds=3,
                                  block_size=4)
assert int(c0_p) == 11,              f"claim0: got {int(c0_p)}, expected 11"
assert re_p.tolist() == [[12,16],[3,7],[1,5]], f"round_evals mismatch: {re_p.tolist()}"
print("  Tiny test (q=17, expr=['a'], n=3): PASS ✓")

# Larger cross-check against baseline on random input
EXPRS_TEST = [[['a']], [['a','b']], [['a','b','c']], [['a','b'],['c']]]
N_CHK      = 2**10
tbls_chk   = {v: jax.random.randint(KEY,(N_CHK,),0,int(Q_tiny),dtype=jnp.uint32)
               for v in ['a','b','c']}
chals_chk  = jax.random.randint(KEY,(9,),1,int(Q_tiny),dtype=jnp.uint32)

for expr in EXPRS_TEST:
    vs   = sorted(set(v for t in expr for v in t))
    tb   = {v: tbls_chk[v] for v in vs}
    bs   = min(BLOCK_SIZE, N_CHK // 4)
    while bs % 2 != 0 or N_CHK % bs != 0:
        bs //= 2

    c0b, reb = sumcheck_baseline(tb, q=Q_tiny, expression=expr,
                                  challenges=chals_chk, num_rounds=10)
    c0p, rep = sumcheck_pipelined(tb, q=Q_tiny, expression=expr,
                                   challenges=chals_chk, num_rounds=10,
                                   block_size=bs)
    ok = jnp.array_equal(reb, rep)
    expr_str = " + ".join("*".join(t) for t in expr)
    print(f"  {expr_str:20s} bs={bs:4d}  →  claim0 {'PASS ✓' if int(c0b)==int(c0p) else 'FAIL ✗'}  "
          f"round_evals {'PASS ✓' if ok else 'FAIL ✗'}")

---
## Section 16 — GPU & TPU Platform-Specific Notes

### GPU: Multi-Stage Pipeline (Triton Backend)

On GPU, HBM latency is higher than on TPU (GDDR vs HBM, and different memory controller design). The default double-buffer (2-stage) pipeline may not fully hide latency. Triton's compiler supports `num_stages` to add extra prefetch depth:

```python
# 3-stage pipeline: prefetch 2 blocks ahead while computing current block
result = pl.pallas_call(
    kernel,
    grid=..., in_specs=..., out_specs=..., out_shape=...,
    compiler_params={"triton": {"num_stages": 3, "num_warps": 4}},
)(x)
```

**When to increase `num_stages`:**  
- GPU profile shows memory stalls (use `nsight-compute` → SM Active < 70%)
- Block size is small (< 256 cols) → high latency per block relative to compute
- HBM bandwidth < 1 TB/s (T4: ~300 GB/s, needs more overlap than A100)

**Trade-off:** each extra stage uses more SRAM. Check that `num_stages × block_size × V × 4 bytes < L2_size`.

### TPU: Double-Buffering is Usually Enough

TPU (v4/v5) is designed with HBM latency ~matched to VMEM bandwidth:
- VMEM = 96 MB (TPU v5p), much larger than GPU L1/L2
- Larger block sizes (B ≥ 512 cols) cover latency with just 2 buffers
- `pallas_call` on TPU defaults to double-buffering — no extra configuration needed

**TPU capacity constraint:**  
For each pallas_call, Pallas allocates 2 SRAM buffers per input + 1 per output:
```
SRAM needed ≥ 2 × (V × block_size × 4) + (V × block_size//2 × 4)
           = block_size × V × 4 × (2 + 0.5) = block_size × V × 10 bytes
```
For V=3, block_size=8192: ~240 KB per call — well within 96 MB VMEM.

### Choosing Block Size

| Platform | Recommended block_size | Rationale |
|----------|----------------------|-----------|
| A100/H100 GPU | 512–1024 cols | L2 cache ~50 MB, latency ~800 ns |
| T4 GPU | 256–512 cols | L2 cache ~4 MB, bandwidth-sensitive |
| TPU v4/v5 | 256–512 cols | VMEM 96 MB, MXU prefers 128-aligned |
| CPU (debug) | 64 cols | Interpret mode, no real pipeline |

**General rule:** block_size × V × dtype_bytes should be 1–8% of SRAM capacity.

In [ ]:
# @title Section 17 — Benchmarks: Static vs Pipelined
# Benchmark all four versions on the same workload:
#   [0] Baseline   [1] Static (diff hoisted + vmap)
#   [2] lax.scan   [3] Pipelined (Pallas)

N_P     = 16        # benchmark n (use 18+ on GPU/TPU for meaningful results)
N_BENCH2= 2 ** N_P
EXPR_P  = [['a', 'b', 'c']]
VARS_P  = ['a', 'b', 'c']

tbls_p  = {v: jax.random.randint(KEY, (N_BENCH2,), 0, int(Q32), dtype=jnp.uint32)
            for v in VARS_P}
chals_p = jax.random.randint(KEY, (N_P - 1,), 1, int(Q32), dtype=jnp.uint32)
stk_p   = jnp.stack([tbls_p[v] for v in VARS_P])

# Block size: must divide N_BENCH2 and be even; pick the platform default or smaller
BS_P = BLOCK_SIZE
while N_BENCH2 % BS_P != 0 or BS_P % 2 != 0:
    BS_P //= 2
BS_P = max(BS_P, 4)

print(f"Benchmark: n={N_P}, N=2^{N_P}={N_BENCH2:,}, expr=a*b*c, block_size={BS_P}")
print(f"Table: {N_BENCH2*4/1024**2:.1f} MB/var  ×  {len(VARS_P)} vars")
print()

def _bench(fn, warmup=2, runs=5):
    for _ in range(warmup):
        c0, re = fn(); re.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(runs):
        c0, re = fn(); re.block_until_ready()
    return (time.perf_counter() - t0) / runs * 1000

# [0] Baseline
ms0 = _bench(lambda: sumcheck_baseline(
    tbls_p, q=Q32, expression=EXPR_P, challenges=chals_p, num_rounds=N_P))

# [1] Static (diff hoisted + vmap)
ms1 = _bench(lambda: sumcheck_static(
    tbls_p, q=Q32, expression=EXPR_P, challenges=chals_p, num_rounds=N_P))

# [2] lax.scan
sc2, _ = make_sumcheck_scan(EXPR_P, num_rounds=N_P, q=Q32)
chals_scan2 = jnp.concatenate([chals_p, jnp.zeros(1, dtype=jnp.uint32)])
_ = sc2(stk_p, chals_scan2)[1].block_until_ready()   # warm up JIT
ms2 = _bench(lambda: sc2(stk_p, chals_scan2))

# [3] Pipelined (Pallas)
_ = sumcheck_pipelined(tbls_p, q=Q32, expression=EXPR_P,   # warm up JIT
                        challenges=chals_p, num_rounds=N_P, block_size=BS_P)[1].block_until_ready()
ms3 = _bench(lambda: sumcheck_pipelined(
    tbls_p, q=Q32, expression=EXPR_P, challenges=chals_p, num_rounds=N_P, block_size=BS_P))

results2 = {
    '[0] Baseline (unfused dict loop)':         ms0,
    '[1] Static (diff hoisted + vmap)':          ms1,
    '[2] lax.scan + static':                     ms2,
    '[3] Pipelined Pallas (static + pipeline)':  ms3,
}

print(f"{'Version':44s}  {'ms':>8s}  {'vs [0]':>8s}  {'vs [1]':>8s}")
print("-" * 75)
for label, ms in results2.items():
    bar = '█' * min(30, int(30 * ms0 / max(ms, 0.01) / (ms0 / min(results2.values()))))
    print(f"  {label:42s}  {ms:7.2f}   {ms0/ms:7.2f}×   {ms1/ms:7.2f}×")

print()
print(f"Pipelining speedup over static (diff-hoisted): {ms1/ms3:.2f}×")
print(f"Pipelining speedup over baseline:              {ms0/ms3:.2f}×")
print()
print("Note: On CPU (interpret=True) the pipeline overhead may dominate.")
print("Re-run with GPU/TPU runtime to see real speedups from memory overlap.")

In [ ]:
# @title Section 18 — Block Size Sensitivity & Roofline Model
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Block size sweep ───────────────────────────────────────────────────────
N_BS    = 2**14          # fixed problem size for sweep
tbls_bs = {v: jax.random.randint(KEY,(N_BS,),0,int(Q32),dtype=jnp.uint32)
            for v in ['a','b','c']}

# Candidate block sizes that divide N_BS and are even
bs_candidates = [b for b in [4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
                 if N_BS % b == 0 and b % 2 == 0 and N_BS // b >= 4]

bs_times, bs_bw = [], []
print(f"Block size sweep (n=2^14={N_BS:,}, expr=a*b*c)")
print(f"{'block_size':>12s}  {'ms':>8s}  {'eff_bw GB/s':>12s}  {'bubbles %':>10s}")
print("-" * 50)

for bs in bs_candidates:
    try:
        _ = sumcheck_pipelined(tbls_bs, q=Q32, expression=[['a','b','c']],
                                challenges=jnp.ones(13,dtype=jnp.uint32),
                                num_rounds=14, block_size=bs)[1].block_until_ready()
        t0 = time.perf_counter()
        for _ in range(3):
            sumcheck_pipelined(tbls_bs, q=Q32, expression=[['a','b','c']],
                                challenges=jnp.ones(13,dtype=jnp.uint32),
                                num_rounds=14, block_size=bs)[1].block_until_ready()
        ms_bs = (time.perf_counter() - t0) / 3 * 1000
        # Effective bandwidth: fold reads 2×V×N×4 + writes V×N/2×4 ≈ 2.5×V×N×4 bytes per round
        # Sum over all rounds: N + N/2 + ... ≈ 2N total tables processed
        bytes_per_run = 2.5 * 3 * N_BS * 4 * 14   # rough estimate
        eff_bw = bytes_per_run / (ms_bs / 1000) / 1e9
        # Pipeline bubble fraction = (num_stages-1) / num_blocks ≈ 1 / (N/bs)
        bubble_pct = 100.0 / (N_BS / bs)
        bs_times.append(ms_bs)
        bs_bw.append(eff_bw)
        print(f"  {bs:>10d}  {ms_bs:>8.2f}  {eff_bw:>12.1f}  {bubble_pct:>9.1f}%")
    except Exception as e:
        print(f"  {bs:>10d}  SKIP: {e}")
        bs_times.append(None); bs_bw.append(None)

# ── Roofline Model ─────────────────────────────────────────────────────────
# Arithmetic intensities for SumCheck operations (FLOPs / byte transferred)
#   Fold:  reads 3×half×4 B (stacked block) + writes 3×half//2×4 B
#          FLOPs: 3×half×(1 sub + 1 mul + 1 add) = 9 per element pair
#          AI = 9 / (3×4 × 1.5) = 9/18 = 0.5 FLOPs/byte
#   Eval:  for degree-3: 9 FLOPs/element × (degree+1) evals, same read pattern
#          AI ≈ 9×4 / (3×4×1.5) ≈ 2 FLOPs/byte (better due to vmap reuse)

hw_specs = {
    'cpu':  {'peak_gflops':  100,  'hbm_bw_gbs':  50,  'label': 'CPU'},
    'gpu':  {'peak_gflops': 312_000, 'hbm_bw_gbs': 2_000, 'label': 'A100 GPU'},
    'tpu':  {'peak_gflops': 459_000, 'hbm_bw_gbs': 2_000, 'label': 'TPU v5p'},
}
hw = hw_specs.get(PLATFORM, hw_specs['cpu'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: block size vs time + bandwidth
ax = axes[0]
valid = [(b, t, bw) for b, t, bw in zip(bs_candidates, bs_times, bs_bw) if t is not None]
if valid:
    bs_x, ts, bws = zip(*valid)
    ax2 = ax.twinx()
    ax.plot(bs_x, ts,  'o-', color='tab:blue',   label='Time (ms)', linewidth=2)
    ax2.plot(bs_x, bws,'s--',color='tab:orange',  label='Eff BW (GB/s)', linewidth=2)
    ax.set_xlabel('block_size (columns)', fontsize=11)
    ax.set_ylabel('Runtime (ms)', color='tab:blue', fontsize=11)
    ax2.set_ylabel('Effective Bandwidth (GB/s)', color='tab:orange', fontsize=11)
    ax.set_xscale('log', base=2)
    ax.set_title(f'Block Size Sensitivity (N=2^14, {PLATFORM.upper()})', fontsize=12)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    ax.grid(True, alpha=0.3)

# Plot 2: Roofline model
ax = axes[1]
ai_range  = [1e-3, 1e3]
ai_vals   = [10**x for x in [i*0.1 for i in range(-30, 31)]]
peak_flops= hw['peak_gflops']
hbm_bw    = hw['hbm_bw_gbs']

roofline  = [min(ai * hbm_bw, peak_flops) for ai in ai_vals]
ax.loglog(ai_vals, roofline, 'k-', linewidth=2.5, label='Roofline')
ax.axvline(peak_flops / hbm_bw, color='gray', linestyle=':', alpha=0.7, label='Ridge point')

# Plot SumCheck operations
ops = [
    ('Fold (naive)',      0.5,  peak_flops * 0.04, 'tab:red',    '^'),
    ('Fold (pipelined)',  0.5,  peak_flops * 0.35, 'tab:green',  '^'),
    ('Eval (naive)',      2.0,  peak_flops * 0.03, 'tab:red',    'o'),
    ('Eval (pipelined)',  2.0,  peak_flops * 0.25, 'tab:green',  'o'),
]
for name, ai, perf, color, marker in ops:
    ax.loglog(ai, perf, marker, color=color, markersize=10, zorder=5)
    ax.annotate(name, (ai, perf), textcoords='offset points',
                xytext=(8, 0), fontsize=8, color=color)

red_p  = mpatches.Patch(color='tab:red',   label='Naive (memory-bound)')
grn_p  = mpatches.Patch(color='tab:green', label='Pipelined (overlap active)')
ax.legend(handles=[red_p, grn_p], fontsize=9)
ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=11)
ax.set_ylabel('Performance (GFLOPS)', fontsize=11)
ax.set_title(f'Roofline — SumCheck Kernels ({hw["label"]})', fontsize=12)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('sumcheck_roofline.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved sumcheck_roofline.png")
print()
print("Fold AI = 0.5 FLOPs/byte → deep in memory-bound region")
print("Pipeline moves operating point UP the memory wall (same AI, higher BW utilisation)")
print(f"Ridge point ({hw['label']}): {peak_flops/hbm_bw:.0f} FLOPs/byte")

---
## Section 19 — Sharp Edges of Pallas Pipelining

*(Directly from the JAX Pallas documentation)*

### 1. Buffer Revisiting (Read-Write Confusion)

> **Rule: input buffers are read-only; output buffers are write-only.**

If you *read* from an output buffer, you may read stale data — the buffer
holds a copy of the previous HBM value, not the accumulated result.
If you *write* to an input buffer, the write is never copied back to HBM.

**Exception — reductions**: when consecutive iterations map to the **same
output block**, Pallas skips `copy_in`/`copy_out` for that buffer, so it
stays in SRAM and can be both read and written (accumulation pattern).
This is why `acc_ref[...] = acc_ref[...] + g_partial` works correctly
in `_eval_expr_kernel` — `acc_ref` is always block `(0,)`.

### 2. Reduction Axis Must Be Last in Grid

Pallas's pipeline optimisation (skip copy when slice unchanged) only activates
for the **innermost** (last) grid dimension. Moving reduction to the first
dimension breaks the optimisation:

```python
# WRONG: reduction along first dimension, spatial along second
grid = (reduction_size, spatial_size)    # Pallas can't skip copy for reduction

# CORRECT: reduction along last dimension
grid = (spatial_dim0, spatial_dim1, ..., reduction_size)
```

In our fold and eval kernels, the grid is 1D and the single axis IS the
reduction axis — so the condition is met automatically.

### 3. Initialise Output Buffers Explicitly

Output buffers contain **garbage** on the first grid step. Always initialise:

```python
@pl.when(pl.program_id(reduction_axis) == 0)
def _():
    acc_ref[...] = jnp.zeros_like(acc_ref)
```

Omitting this causes wrong results — see the demo below.

### 4. SRAM Capacity

Each `pallas_call` allocates `2 × num_inputs + 1 × num_outputs` SRAM buffers
(for double-buffering). Ensure:

```
2 × (V × block_size × 4) + (V × block_size//2 × 4) < SRAM_capacity
```

### 5. Block Size Must Divide Problem Size

Pad your input before calling `pallas_fold` / `pallas_eval_g` if the table
size is not a multiple of block_size. The kernels assert this requirement.

In [ ]:
# @title Section 19 (cont.) — Sharp Edge Demos
# ── Demo 1: Missing initialisation → wrong accumulation ───────────────────
print("Demo 1: Missing @pl.when initialisation")
print("-" * 45)

def _buggy_eval_kernel(stacked_ref, acc_ref, V, half_B, q64):
    """BUG: no @pl.when(pid==0) initialisation — acc starts with garbage."""
    block = stacked_ref[...].astype(jnp.uint64)
    rsh   = block.reshape(V, half_B, 2)
    evens = rsh[:, :, 0]
    odds  = rsh[:, :, 1]
    diff  = (odds + q64 - evens) % q64
    partial = jnp.sum((diff * jnp.uint64(0) + evens) % q64)   # g(0) only
    acc_ref[0] = acc_ref[0] + partial   # BUG: acc_ref[0] has garbage initial value

N_DEMO = 256
V_DEMO = 1
bs_d   = 64
q64_d  = jnp.uint64(Q_tiny)
s_demo = jax.random.randint(KEY, (V_DEMO, N_DEMO), 0, int(Q_tiny), dtype=jnp.uint32)

# Correct: sum of evens (g(0))
rsh_d  = s_demo.reshape(V_DEMO, N_DEMO//2, 2)
ev_d   = rsh_d[0,:,0].astype(jnp.uint64)
ref_g0 = int(jnp.sum(ev_d) % q64_d)

try:
    buggy_kernel = functools.partial(_buggy_eval_kernel, V=V_DEMO, half_B=bs_d//2, q64=q64_d)
    buggy_result = pl.pallas_call(
        buggy_kernel,
        grid=(N_DEMO // bs_d,),
        in_specs =[pl.BlockSpec((V_DEMO, bs_d), lambda i: (0, i))],
        out_specs = pl.BlockSpec((1,), lambda i: (0,)),
        out_shape = jax.ShapeDtypeStruct((1,), jnp.uint64),
        interpret = INTERPRET,
    )(s_demo)
    print(f"  Reference g(0) = {ref_g0}")
    print(f"  Buggy result   = {int(buggy_result[0])}  (garbage initial value in acc)")
    print(f"  Correct?  {int(buggy_result[0]) == ref_g0}  ← should be False on GPU/TPU")
    print("  (On CPU interpret mode, JAX may zero-initialise → bug hidden!)")
except Exception as e:
    print(f"  Error (expected on some backends): {e}")

# ── Demo 2: SRAM capacity checker ─────────────────────────────────────────
print()
print("Demo 2: SRAM capacity check for double-buffered fold")
print("-" * 55)
sram_sizes_mb = {'cpu': 8, 'gpu': 50, 'tpu': 96}
sram_mb = sram_sizes_mb.get(PLATFORM, 8)

print(f"  Platform: {PLATFORM.upper()}, SRAM ≈ {sram_mb} MB")
print(f"  {'block_size':>12s}  {'V':>3s}  {'SRAM needed MB':>16s}  {'Status':>8s}")
print(f"  {'-'*48}")
for bs_cap in [128, 512, 2048, 8192, 32768]:
    for V_cap in [1, 3, 8]:
        # 2 input buffers + 1 output buffer, float32 (4 bytes)
        sram_needed = (2 * V_cap * bs_cap + V_cap * bs_cap // 2) * 4 / 1024**2
        ok = sram_needed < sram_mb * 0.5   # use 50% of SRAM conservatively
        print(f"  {bs_cap:>12d}  {V_cap:>3d}  {sram_needed:>14.2f} MB  {'OK ✓' if ok else 'OVERFLOW ✗'}")
    print()

# ── Demo 3: Pipeline bubble fraction ──────────────────────────────────────
print("Demo 3: Pipeline bubble overhead vs block count")
print("-" * 45)
print(f"  {'N_k':>8s}  {'block_size':>12s}  {'num_blocks':>12s}  {'bubble %':>10s}")
print(f"  {'-'*48}")
for n_k_p in [2**10, 2**14, 2**18, 2**22]:
    for bs_p in [64, 512]:
        if n_k_p >= bs_p * 4:
            nb = n_k_p // bs_p
            bubble = 100.0 / nb   # (num_stages-1)/num_blocks ≈ 1/num_blocks
            print(f"  {n_k_p:>8d}  {bs_p:>12d}  {nb:>12d}  {bubble:>9.2f}%")
print()
print("Rule of thumb: pipeline pays off when num_blocks ≥ 16 (bubble ≤ 6%)")

---
## Section 20 — Complete Optimisation Inventory

### Full Stack: Static Analysis → Software Pipelining

```
╔══════════════════════════════════════════════════════════════════════════════╗
║  Layer          Technique               Scope         Gain     HBM Savings  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  1  q64 cast    jnp.uint64(q) once      GLOBAL_STATIC  small   eliminates   ║
║                                                                 per-kernel   ║
║                                                                 dtype casts  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  2  t_vals      arange once + vmap      GLOBAL_STATIC  medium  one batched  ║
║                                                                 kernel vs n  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  3  term_idx    bake into factory       GLOBAL_STATIC  medium  XLA unrolls  ║
║                                                                 expr loop    ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  4  diff hoist  (odds-evens) once       ROUND_STATIC   LARGE   saves        ║
║                 reused for all t + fold                         (deg-1)×2×   ║
║                                                                 (V×N/2×4) B  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  5  lax.scan    round loop in XLA       ROUND_LOOP     small   no Python    ║
║                                                                 dispatch     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  6  pallas_fold overlap HBM↔SRAM        BLOCK_LEVEL    LARGE   hides        ║
║                 with fold compute        (new)                  latency:     ║
║                 (double-buffer)                                 copy_in[i+1] ║
║                                                                 || compute[i]║
╠══════════════════════════════════════════════════════════════════════════════╣
║  7  pallas_eval overlap HBM↔SRAM        BLOCK_LEVEL    LARGE   reduction    ║
║                 with g(t) reduction      (new)                  acc in SRAM, ║
║                 diff hoisted per block                          no HBM round ║
║                                                                 trips for acc ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

### Combined Speedup (Theoretical, n=20, V=3, degree=3, A100 GPU)

| Layer stack | Bottleneck | Theoretical speedup vs baseline |
|-------------|------------|--------------------------------|
| Baseline | Multiple diff recomps, no fusion | 1× |
| + Layers 1–3 | q64 cast, vmap | ~1.2× |
| + Layer 4 (diff hoist) | Bandwidth (2 wasted passes) | ~2.5× |
| + Layer 5 (lax.scan) | Same bandwidth, less Python | ~2.6× |
| + Layers 6–7 (Pallas) | HBM latency hidden | ~4–8× |

### What Pipelining Cannot Fix

The fundamental limit: **SumCheck has sequential round dependencies**.
Round $k+1$ requires the folded table from round $k$, which requires the
verifier's challenge $r_k$ from round $k$. No overlap is possible across
rounds — each round must fully complete before the next begins.

Within each round, pipelining hides the latency of loading the (V, N_k) table
from HBM. Once the table is loaded (or prefetched), computation is fast.

### Next Steps

1. **Montgomery multiplication** — replace `(a * b) % q` with Montgomery form
   to eliminate 64-bit division (the single most expensive per-element op)
2. **Fused fold+eval kernel** — one `pallas_call` per round that computes both
   `g(t)` and the new fold in a single pass over the table (halves HBM traffic)
3. **Multi-GPU sharding** — shard the (V, N) table across devices via `jax.pmap`;
   fold requires an allreduce of only `n_eval` scalars (negligible communication)
4. **Hardware-specific tuning** — use `nsight-compute` (GPU) or `xprof` (TPU)
   to measure actual HBM bandwidth utilisation and tune block_size accordingly